# Roman-like injection–recovery

Fit a finite-source event with **annual parallax, limb darkening, and
correlated stellar variability**. We will write the probability model in
NumPyro, sample it with NUTS, and inspect the posterior light curve,
residuals, corner plot, and chains.

Assume `times`, `flux`, and `flux_error` are one-dimensional arrays of
observation times, measured flux, and known Gaussian flux errors. Times
must be sorted in increasing order. Here they are HJD − 2458000 in days;
fluxes use a magnitude zeropoint of 25.

The example has a 72-day season at 12-minute cadence, giving 8640 points.
Its W149 source magnitude is 24.744 and source fraction is 0.395, following
the [reference Roman experiment](https://github.com/ShotaMiyazaki94/test_microjax/tree/979fa9b33b69c4ba89eb7ad70ebae3d23da349cc/Roman_recovery).
We assume an **active low-mass dwarf** with 1% intrinsic flux RMS, a 5-day
rotation period, and 20-day coherence. Near-infrared spot modulation in
active M dwarfs motivates this scale
([Goulding et al. 2012](https://doi.org/10.1111/j.1365-2966.2012.21932.x));
the variability class and coherence are scenario choices, not consequences
of apparent magnitude alone. This is an idealized example, not a Roman
performance forecast.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
from example._common import style
style()


In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import singlens as sl


In [ ]:
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_median
from tinygp import GaussianProcess, kernels

geometry = sl.AnnualParallax(ra_deg=266.0, dec_deg=-29.0, t_ref=0.0, time_origin=2458000.0)


In [ ]:
from example.inference.code.inference import load_injection

_, injection = load_injection()
times, flux, flux_error = injection.times, injection.flux, injection.error
truth = injection.truth
true_lens_flux = np.asarray(injection.lens_flux)
true_observed_flux = np.asarray(injection.lens_flux + injection.gp_signal)


## Source variability and the likelihood

Let $A(t)$ be the lens magnification and $g(t)$ the intrinsic source-flux
variation. The observation model is

$$F(t)=A(t)[f_s+g(t)]+f_b+\epsilon(t).$$

The lens magnifies the variability too. We use a damped-oscillator GP with
period $P$, coherence time $\tau$, and source-flux RMS $\sigma$:

$$k(\Delta t)=\sigma^2 e^{-|\Delta t|/\tau}
\left[\cos\left(\frac{2\pi|\Delta t|}{P}\right)
+\frac{P}{2\pi\tau}\sin\left(\frac{2\pi|\Delta t|}{P}\right)\right].$$

In tinygp's SHO convention, the undamped frequency is
$\omega_0=\sqrt{(2\pi/P)^2+\tau^{-2}}$ and $Q=\omega_0\tau/2$.
`gp_sigma` is in **unmagnified source-flux units**, while `gp_scale`
and `gp_period` are in days.

Blending dilutes the 1% source RMS to about 4.3 mmag at baseline,
approaching 10.9 mmag at high magnification. This disk-integrated model
does not describe individual spot crossings.


In [ ]:
def source_gp(times, errors, sigma, scale, period):
    omega = jnp.hypot(2 * jnp.pi / period, 1 / scale)
    kernel = kernels.quasisep.SHO(omega=omega, quality=0.5 * omega * scale, sigma=sigma)
    return GaussianProcess(kernel, times, diag=errors**2)


Marginalize the latent GP rather than sampling a value at every time.
In source-flux units the residual and white-noise error are
$r_i=(F_i-f_b)/A_i-f_s$ and $\sigma_i/A_i$. Converting this density back
to observed flux contributes $-\sum_i\log A_i$ to the log likelihood.
Keeping that term matters because $A_i$ changes with the event parameters.

The following NumPyro model contains all twelve inferred parameters.
Positive quantities have log-normal priors; `a1` has a logit-normal prior
on $(0,1)$. These distributions match the priors used for the recovery
below. This example selects the `u0 > 0` branch.


In [ ]:
def microlensing_model(times, flux, flux_error):
    t0 = numpyro.sample("t0", dist.Normal(0.0, 0.5))
    u0 = numpyro.sample("u0", dist.LogNormal(jnp.log(0.01), 1.0))
    t_e = numpyro.sample("t_e", dist.LogNormal(jnp.log(15.0), 0.6))
    rho = numpyro.sample("rho", dist.LogNormal(jnp.log(0.01), 0.7))
    f_s = numpyro.sample("f_s", dist.LogNormal(0.0, 1.0))
    f_b = numpyro.sample("f_b", dist.Normal(0.0, 5.0))
    pi_e_n = numpyro.sample("pi_e_n", dist.Normal(0.0, 0.5))
    pi_e_e = numpyro.sample("pi_e_e", dist.Normal(0.0, 0.5))
    a1 = numpyro.sample(
        "a1",
        dist.TransformedDistribution(dist.Normal(0.0, 1.0), dist.transforms.SigmoidTransform()),
    )
    gp_sigma = numpyro.sample("gp_sigma", dist.LogNormal(jnp.log(0.01), 1.0))
    gp_scale = numpyro.sample("gp_scale", dist.LogNormal(jnp.log(20.0), 0.7))
    gp_period = numpyro.sample("gp_period", dist.LogNormal(jnp.log(5.0), 0.7))

    A = sl.magnification(
        times,
        t0,
        u0,
        t_e,
        rho=rho,
        a1=a1,
        parallax=geometry,
        pi_e_n=pi_e_n,
        pi_e_e=pi_e_e,
    )
    residual = (flux - f_b) / A - f_s
    gp = source_gp(times, flux_error / A, gp_sigma, gp_scale, gp_period)
    numpyro.factor("flux_loglikelihood", gp.log_probability(residual) - jnp.log(A).sum())


## Sample with NUTS

Pass the probability model to NumPyro. The two chains run sequentially on
CPU. Warmup adapts step size and a dense mass matrix; retained samples
contain event parameters, both fluxes, limb darkening, and the three GP
hyperparameters. Initial draws near the prior medians avoid starting
positive scale parameters far outside their expected ranges.


In [ ]:
kernel = NUTS(
    microlensing_model,
    init_strategy=init_to_median(num_samples=50),
    dense_mass=True,
    target_accept_prob=0.9,
)
mcmc = MCMC(
    kernel,
    num_warmup=500,
    num_samples=800,
    num_chains=2,
    chain_method="sequential",
    progress_bar=False,
)
mcmc.run(jax.random.key(18), times, flux, flux_error)
draws = mcmc.get_samples(group_by_chain=True)
mcmc.print_summary()
print("Divergences:", int(mcmc.get_extra_fields()["diverging"].sum()))


Keep the chain axis when assessing convergence: each `draws[name]` has
shape `(chains, retained draws)`. Inspect R̂, effective sample sizes, and
divergences before interpreting the posterior. Increase warmup or improve
initialization if those diagnostics are unsatisfactory.

The results below use the full saved 8640-point recovery: two chains,
500 warmup steps, and 800 retained draws per chain. Its priors and
likelihood are the ones written above. The command-line example also
initializes from a joint fit and rescales coordinates for sampling.


In [ ]:
import json
from example.inference.code.inference import OUTPUT, NAMES, data_hash

if "draws" not in globals():
    with np.load(OUTPUT / "posterior.npz") as saved:
        draws = {name: saved[name] for name in NAMES}
    diagnostics = json.loads((OUTPUT / "hmc.json").read_text())
    assert diagnostics["data_sha256"] == data_hash(injection)
    divergences = diagnostics["divergences"]
else:
    divergences = int(mcmc.get_extra_fields()["diverging"].sum())


In [ ]:
from numpyro.diagnostics import summary

diagnostics_by_parameter = summary(draws, group_by_chain=True)
print("Maximum R-hat:", max(s["r_hat"] for s in diagnostics_by_parameter.values()))
print("Minimum ESS:", min(s["n_eff"] for s in diagnostics_by_parameter.values()))


In [ ]:
from html import escape
from IPython.display import HTML, Markdown, display

display(Markdown(f"Divergences: **{divergences}**."))
table = [
    '<table class="table"><thead><tr><th>Parameter</th><th>Injected</th>'
    "<th>Median</th><th>Central 90% interval</th></tr></thead><tbody>"
]
for name, values in draws.items():
    low, median, high = np.quantile(values, [0.05, 0.5, 0.95])
    table.append(
        f"<tr><th><code>{escape(name)}</code></th>"
        f"<td>{truth[name]:.6g}</td><td>{median:.6g}</td>"
        f"<td>[{low:.6g}, {high:.6g}]</td></tr>"
    )
table.append("</tbody></table>")
display(HTML("".join(table)))


## Posterior light curve and residuals

To include uncertainty in the stellar variability, condition the source
GP at each joint parameter draw and sample a complete correlated path.
Adding only the GP mean would omit part of the predictive uncertainty.
The code below predicts the **latent lens + source-variability flux**;
it does not add new measurement noise.


In [ ]:
def posterior_flux(params, key):
    A = sl.magnification(
        times,
        params["t0"],
        params["u0"],
        params["t_e"],
        rho=params["rho"],
        a1=params["a1"],
        parallax=geometry,
        pi_e_n=params["pi_e_n"],
        pi_e_e=params["pi_e_e"],
    )
    lens_flux = params["f_s"] * A + params["f_b"]
    gp = source_gp(
        times,
        flux_error / A,
        params["gp_sigma"],
        params["gp_scale"],
        params["gp_period"],
    )
    conditional = gp.condition((flux - lens_flux) / A, diag=0.0).gp
    return lens_flux + A * conditional.sample(key)


flat_draws = {name: jnp.asarray(values).reshape(-1) for name, values in draws.items()}
keys = jax.random.split(jax.random.key(23), flat_draws["t0"].size)
flux_paths = jax.jit(lambda p, k: jax.lax.map(lambda pair: posterior_flux(*pair), (p, k)))(
    flat_draws, keys
)
magnitude_paths = 25.0 - 2.5 * np.log10(np.asarray(flux_paths))
low, median, high = np.quantile(magnitude_paths, [0.05, 0.5, 0.95], axis=0)


For an injection, the true lens-only flux `true_lens_flux` provides a
useful residual baseline:

$$\Delta m=-2.5\log_{10}(F_{\rm observed}/F_{\rm lens,true}).$$

Both the simulated GP and white noise remain in these residuals. The
posterior band includes the event, flux, GP hyperparameter, and latent-GP
uncertainty. We also show inverse-variance residual means in 0.1-day bins,
shorter than the 5-day rotation period and 20-day coherence time.
Binning is for display; all 8640 original observations enter the fit.


In [ ]:
observed_mag = 25.0 - 2.5 * np.log10(np.asarray(flux))
true_mag = 25.0 - 2.5 * np.log10(true_lens_flux)
residual_mag = observed_mag - true_mag
# Propagate the known white noise at the noiseless injected flux.
sigma_mag = 2.5 / np.log(10.0) * np.asarray(flux_error) / true_observed_flux
weights = 1 / sigma_mag**2
bins = np.floor(np.round((np.asarray(times) - float(times[0])) / 0.1, 9)).astype(int)
total_weight = np.bincount(bins, weights=weights)
occupied = total_weight > 0
bin_time = np.bincount(bins, weights=weights * np.asarray(times))[occupied] / total_weight[occupied]
bin_residual = np.bincount(bins, weights=weights * residual_mag)[occupied] / total_weight[occupied]
bin_error = 1 / np.sqrt(total_weight[occupied])

fig, axes = plt.subplots(
    2, 1, figsize=(14, 6), sharex=True, height_ratios=[2, 1], layout="constrained"
)
axes[0].plot(times, observed_mag, ".", ms=1.5, alpha=0.3, color="#83919C", label="Data")
axes[0].plot(times, true_mag, "--", lw=1.0, color="#E07A5F", label="True lens")
axes[0].plot(times, median, color="#247BA0", label="Model + GP median")
axes[0].fill_between(times, low, high, color="#247BA0", alpha=0.25, label="90% interval")
axes[0].set_ylabel("W149 [mag]")
axes[0].invert_yaxis()
axes[0].legend(ncol=2)
axes[1].axhline(0.0, color="0.4", lw=0.8, ls="--")
axes[1].plot(times, residual_mag, ".", ms=1.5, alpha=0.2, color="#83919C")
axes[1].errorbar(
    bin_time,
    bin_residual,
    yerr=bin_error,
    fmt=".",
    ms=2.0,
    elinewidth=0.45,
    color="#37474F",
    label="0.1-day bins (1σ)",
)
axes[1].plot(times, median - true_mag, color="#247BA0", label="Model + GP median")
axes[1].fill_between(times, low - true_mag, high - true_mag, color="#247BA0", alpha=0.25)
axes[1].set(xlim=(-36, 36), xlabel="HJD − 2458000 [day]", ylabel="Residual to true lens [mag]")
axes[1].legend(ncol=2)
plt.show()


The bin errors include only white measurement noise, not the GP covariance.
Here the noiseless injected lens + GP flux `true_observed_flux` sets the
flux-to-magnitude error conversion. For real observations, use a fitted
positive flux prediction instead. Magnitude error propagation is a
small-noise approximation.

## Parameter correlations

Keep the twelve parameters together when making a corner plot. Flattening
the chain and draw axes is appropriate for this plot after convergence
checks. `truth` is the dictionary of injected parameter values; omit it
for real observations.


In [ ]:
import corner

names = [
    "t0",
    "u0",
    "t_e",
    "rho",
    "f_s",
    "f_b",
    "gp_sigma",
    "gp_scale",
    "pi_e_n",
    "pi_e_e",
    "a1",
    "gp_period",
]
labels = [
    r"$t_0$ [day]",
    r"$u_0$",
    r"$t_E$ [day]",
    r"$\rho$",
    r"$f_s$",
    r"$f_b$",
    r"$\sigma_{\rm GP}$",
    r"$\tau_{\rm GP}$ [day]",
    r"$\pi_{E,N}$",
    r"$\pi_{E,E}$",
    r"$a_1$",
    r"$P_{\rm rot}$ [day]",
]
values = np.column_stack([np.asarray(draws[name]).reshape(-1) for name in names])
fig = corner.corner(
    values,
    labels=labels,
    truths=[truth[name] for name in names],
    color="#247BA0",
    truth_color="#E07A5F",
    levels=(0.68, 0.95),
    quantiles=[0.05, 0.5, 0.95],
    show_titles=True,
    title_fmt=".4g",
    plot_datapoints=False,
    fill_contours=True,
    max_n_ticks=3,
    title_kwargs={"fontsize": 10},
    label_kwargs={"fontsize": 11},
)
plt.show()


The limb coefficient, source radius, event timescale, and fluxes are
correlated. The north parallax component remains weakly constrained in
this short event; fitting both components does not imply that both are
precisely measured. The rotation period is constrained near 5 days while
the coherence time remains broad. Download the
{download}`full-size corner PDF <../../example/inference/outputs/corner.pdf>`.

## Trace plots

Plot the chains separately before pooling them. A corner plot alone can
hide a chain that stays in a different region of parameter space.


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 6), layout="constrained")
for ax, name, label in zip(axes.flat, names, labels):
    for chain, samples in enumerate(draws[name]):
        ax.plot(samples, lw=0.5, alpha=0.75, label=f"Chain {chain + 1}")
    ax.axhline(truth[name], color="black", lw=0.8, ls="--")
    ax.set_ylabel(label)
    ax.ticklabel_format(axis="y", useOffset=False)
for ax in axes[-1]:
    ax.set_xlabel("Retained draw")
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="outside upper right", ncol=2)
plt.show()


Convergence does not establish coverage. One noise realization can shift
correlated parameters away from their injected values, and this positive-
`u0` fit does not explore the other parallax solutions.

Regenerate the full saved experiment, including data and figures, with
`python example/inference/code/inference.py --hmc` from the repository root.
The [inference benchmark](../benchmarks.md#inference-efficiency) compares
singlens/NUTS and VBM/emcee on a separate nine-parameter event with
independent measurement noise.
